# YOLO Tracking + VLM Two-Stage Road Defect Detection — Dashboard Pipeline

| Step | Description |
|------|-------------|
| 1 | Feed video through **`model.track(persist=True)`** — each defect gets a unique `track_id` |
| 2 | **Visualize YOLO tracking first** — annotated frames + stats shown before any VLM work |
| 3 | **conf > 0.8** → AUTO-ACCEPT → sent to dashboard |
| 4 | **conf < 0.3** → REJECT → sent to dashboard |
| 5 | **0.3 ≤ conf ≤ 0.8** → ROI expand + adaptive shrink → VLM verify |
| 6 | VLM called **once per unique `track_id`** — same defect reuses cached verdict |
| 7 | VLM **YES** → sent to dashboard as accepted detection |
| 8 | VLM **NO** → sent to dashboard as YOLO detection + VLM rejection (with reasoning) |
| 9 | Inline visualizations shown in notebook |

> **All results are sent to the Flask + ngrok dashboard** — no files saved to Drive.
> Start `python app.py` on your local machine first, then paste the ngrok URL below.

In [ ]:
# ──────────────────────────────────────────────────────────────
# 1.1  Install Dependencies
# ──────────────────────────────────────────────────────────────
# Upgrade Pillow first — avoids the _Ink ImportError with ultralytics
!pip install -q "pillow>=10.4.0" --upgrade
!pip install -q "transformers @ git+https://github.com/huggingface/transformers.git@main"
!pip install -q accelerate ultralytics matplotlib seaborn
!pip install -q qwen-vl-utils torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 118.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 26.9 MB/s eta 0:00:00


In [ ]:
# ──────────────────────────────────────────────────────────────
# 1.2  Imports
# ──────────────────────────────────────────────────────────────

!pip install -q "pillow>=10.4.0" --upgrade --force-reinstall

import os, gc, re, json, warnings, base64, requests
from collections import defaultdict

import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from ultralytics import YOLO
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 2.1  Mount Google Drive
# ──────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ──────────────────────────────────────────────────────────────
# 2.2  Configure Paths & Dashboard Connection
# ──────────────────────────────────────────────────────────────

# ── Input video (unannotated) ──
VIDEO_PATH        = "/content/drive/MyDrive/Mashail/vlm/road test.mp4"

# ── YOLO weights ──
YOLO_WEIGHTS      = "/content/drive/MyDrive/Mashail/vlm/01 Training Logs YOLOv11n Dataset V6.2 (2class)/724127398696099981/8d0fa57398f44032bb1842f85ac98130/artifacts/weights/best.pt"

# ── VLM model ──
VLM_MODEL_ID      = "Qwen/Qwen3.5-2B"

# ── Frame sampling ──
FRAME_SAMPLE_RATE = 5

# ══════════════════════════════════════════════════════════════
# DASHBOARD CONNECTION  (Flask + ngrok)
# ══════════════════════════════════════════════════════════════
DASHBOARD_URL = "https://anthracoid-nonmartially-darien.ngrok-free.dev"

# Quick connectivity check
try:
    r = requests.get(f"{DASHBOARD_URL}/api/stats", timeout=10)
    r.raise_for_status()
    print(f"Dashboard connected  : {DASHBOARD_URL}")
except Exception as e:
    print(f"WARNING: Cannot reach dashboard at {DASHBOARD_URL}")
    print(f"  Error: {e}")
    print(f"  Make sure `python app.py` is running and ngrok is active.\n")

print(f"Video         : {VIDEO_PATH}")
print(f"YOLO weights  : {YOLO_WEIGHTS}")
print(f"VLM model     : {VLM_MODEL_ID}")
print(f"Frame rate    : every {FRAME_SAMPLE_RATE} frame(s)")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 3.1  Pipeline Configuration
# ──────────────────────────────────────────────────────────────

class PipelineConfig:

    # ── Confidence thresholds ──
    LOW_THRESH  = 0.30
    HIGH_THRESH = 0.80

    # ── ROI expansion ──
    INITIAL_EXPAND_SCALE = 1.5
    SHRINK_SCALES        = [1.5, 1.4, 1.3, 1.2, 1.1, 1.0]
    OVERLAP_THRESH       = 0.30

    # ── Class mapping ──
    CLASS_NAMES = {0: 'crack', 1: 'pothole'}
    CLASS_IDS   = {'crack': 0, 'pothole': 1}

    # ====================================================
    # YES/NO Prompts  (thinking OFF -- fast verification)
    # ====================================================
    PROMPTS = {
        'crack': (
            "You are a road defect inspector. "
            "Look at this cropped road surface image.\n\n"
            "Is there ANY type of crack visible on the road surface? "
            "This includes: longitudinal cracks, transverse cracks, "
            "alligator/fatigue cracking, block cracking, edge cracks, "
            "hairline cracks, or any linear fracture pattern.\n\n"
            "Answer EXACTLY in this format:\n"
            "ANSWER: YES or NO\n\n"
            "Important: If you see ANY form of cracking, answer YES."
        ),
        'pothole': (
            "You are a road defect inspector. "
            "Look at this cropped road surface image.\n\n"
            "Is there ANY type of pothole visible on the road surface? "
            "This includes: bowl-shaped potholes, shallow depressions, "
            "deep holes, irregular cavities, surface break-outs, "
            "or any area where pavement material has broken away.\n\n"
            "Answer EXACTLY in this format:\n"
            "ANSWER: YES or NO\n\n"
            "Important: If you see ANY form of pothole or pavement break-out, answer YES."
        ),
    }

    # ====================================================
    # Reasoning Prompts  (thinking ON -- only on NO verdicts)
    # ====================================================
    REASONING_PROMPTS = {
    'crack': (
        "You are a road defect inspector. "
            "Look at this cropped road surface image.\n\n"
        "Describe what is visible on the road surface in one short sentence."
    ),
    'pothole': (
        "You are a road defect inspector. "
            "Look at this cropped road surface image.\n\n"
        "Describe what is visible on the road surface in one short sentence."
    ),
    }

    # ── VLM generation parameters ──
    VLM_TEMPERATURE       = 0.6
    VLM_TOP_P             = 0.95
    VLM_TOP_K             = 20
    VLM_MAX_TOKENS_VERIFY = 30
    VLM_MAX_TOKENS_REASON = 100

    # ── Visualization ──
    MAX_VIS_PER_CATEGORY  = 15
    MAX_TRACKING_FRAMES   = 20

cfg = PipelineConfig()
print("Pipeline configuration loaded.")
print(f"  LOW_THRESH  : {cfg.LOW_THRESH}  (below -> reject)")
print(f"  HIGH_THRESH : {cfg.HIGH_THRESH}  (above -> auto-accept)")
print(f"  VLM zone    : [{cfg.LOW_THRESH} - {cfg.HIGH_THRESH}]")
print(f"  Frame rate  : every {FRAME_SAMPLE_RATE} frame(s)")

Pipeline configuration loaded.
  LOW_THRESH  : 0.3  (below -> reject)
  HIGH_THRESH : 0.8  (above -> auto-accept)
  VLM zone    : [0.3 - 0.8]
  Frame rate  : every 5 frame(s)


In [ ]:
# ──────────────────────────────────────────────────────────────
# 4.1  Utility Functions
# ──────────────────────────────────────────────────────────────

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def compute_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    union = areaA + areaB - inter
    return inter / union if union > 0 else 0.0


def overlap_ratio(boxA, boxB):
    xA = max(boxA[0], boxB[0]); yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]); yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / areaB if areaB > 0 else 0.0


def adjust_to_square_bbox(x1, y1, x2, y2, W, H, scale):
    bbw, bbh = x2 - x1, y2 - y1
    L  = max(bbw * scale, bbh * scale)
    cx, cy = x1 + bbw / 2.0, y1 + bbh / 2.0
    return (max(0, int(cx - L/2.0)), max(0, int(cy - L/2.0)),
            min(W, int(cx + L/2.0)), min(H, int(cy + L/2.0)))


def adaptive_expand_bbox(x1, y1, x2, y2, W, H, all_expanded_rois, current_idx):
    for s in cfg.SHRINK_SCALES:
        ex1, ey1, ex2, ey2 = adjust_to_square_bbox(x1, y1, x2, y2, W, H, s)
        expanded   = [ex1, ey1, ex2, ey2]
        overlap_ok = True
        for idx, other_roi in enumerate(all_expanded_rois):
            if idx == current_idx or other_roi is None:
                continue
            if overlap_ratio(expanded, other_roi) > cfg.OVERLAP_THRESH:
                overlap_ok = False
                break
        if overlap_ok:
            return ex1, ey1, ex2, ey2
    return x1, y1, x2, y2


def get_yolo_tracked_boxes(img):
    results = yolo_model.track(img, persist=True, verbose=False)[0]
    boxes   = []
    if results.boxes is None or results.boxes.id is None:
        return boxes
    for r, tid in zip(results.boxes, results.boxes.id):
        x1, y1, x2, y2 = map(int, r.xyxy[0])
        conf     = float(r.conf[0])
        cls      = int(r.cls[0])
        track_id = int(tid)
        boxes.append({
            'bbox'      : [x1, y1, x2, y2],
            'class'     : cfg.CLASS_NAMES[cls],
            'class_id'  : cls,
            'confidence': conf,
            'track_id'  : track_id,
        })
    return boxes


# ──────────────────────────────────────────────────────────────
# Dashboard HTTP helpers
# ──────────────────────────────────────────────────────────────

def encode_image_b64(img_bgr):
    """Encode a BGR numpy image as base64 JPEG."""
    _, buf = cv2.imencode('.jpg', img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 85])
    return base64.b64encode(buf).decode('utf-8')


def crop_bbox(img, bbox, pad=20):
    """Crop image around bbox with padding."""
    h, w = img.shape[:2]
    x1, y1, x2, y2 = bbox
    return img[max(0,y1-pad):min(h,y2+pad), max(0,x1-pad):min(w,x2+pad)]


def send_yolo_detection(defect_type, confidence, model_name, image_bgr):
    """POST a YOLO detection to the dashboard. Returns detection_id or None."""
    payload = {
        "defect_type": defect_type,
        "confidence": round(confidence, 4),
        "model_name": model_name,
        "image": encode_image_b64(image_bgr),
    }
    try:
        r = requests.post(f"{DASHBOARD_URL}/yolo-inference", json=payload, timeout=15)
        r.raise_for_status()
        det_id = r.json().get("detection_id")
        return det_id
    except Exception as e:
        print(f"  [WARN] /yolo-inference failed: {e}")
        return None


def send_vlm_no(defect_type, model_name, reasoning, image_bgr, detection_id=None):
    """POST a VLM rejection to the dashboard."""
    payload = {
        "defect_type": defect_type,
        "model": model_name,
        "reasoning": reasoning,
        "image": encode_image_b64(image_bgr),
    }
    if detection_id is not None:
        payload["detection_id"] = detection_id
    try:
        r = requests.post(f"{DASHBOARD_URL}/vlm-no", json=payload, timeout=15)
        r.raise_for_status()
    except Exception as e:
        print(f"  [WARN] /vlm-no failed: {e}")


print("Utility functions loaded.")
print("  get_yolo_tracked_boxes() -> uses model.track(persist=True)")
print("  send_yolo_detection()    -> POST /yolo-inference")
print("  send_vlm_no()            -> POST /vlm-no")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 5.1  Load YOLO Model
# ──────────────────────────────────────────────────────────────
yolo_model = YOLO(YOLO_WEIGHTS)
print(f"YOLO loaded : {YOLO_WEIGHTS}")

YOLO loaded : /content/drive/MyDrive/Mashail/vlm/01 Training Logs YOLOv11n Dataset V6.2 (2class)/724127398696099981/8d0fa57398f44032bb1842f85ac98130/artifacts/weights/best.pt


In [ ]:
# ──────────────────────────────────────────────────────────────
# 5.2  Load Qwen3.5-2B VLM
# ──────────────────────────────────────────────────────────────
from transformers import AutoModelForImageTextToText, AutoProcessor

vlm_model = AutoModelForImageTextToText.from_pretrained(
    VLM_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
vlm_processor = AutoProcessor.from_pretrained(VLM_MODEL_ID)

if vlm_processor.tokenizer.pad_token_id is None:
    vlm_processor.tokenizer.pad_token_id = vlm_processor.tokenizer.eos_token_id

print(f"VLM loaded  : {VLM_MODEL_ID}")
print(f"  Verify calls   : thinking OFF (YES/NO only)")
print(f"  Reasoning calls: thinking ON  (NO verdicts only)")

config.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/4.55G [00:00<?, ?B/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


...

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

VLM loaded  : Qwen/Qwen3.5-2B
  Verify calls   : thinking OFF (YES/NO only)
  Reasoning calls: thinking ON  (NO verdicts only)


In [ ]:
# ──────────────────────────────────────────────────────────────
# 6.1  VLM Inference  --  Verify + Reason
# ──────────────────────────────────────────────────────────────

def vlm_query(pil_image, prompt_text, enable_thinking=False, max_tokens=30):
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': pil_image},
        {'type': 'text',  'text': prompt_text},
    ]}]
    inputs = vlm_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
        enable_thinking=enable_thinking,
    ).to(vlm_model.device)

    with torch.no_grad():
        generated_ids = vlm_model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=cfg.VLM_TEMPERATURE,
            top_p=cfg.VLM_TOP_P,
            top_k=cfg.VLM_TOP_K,
            do_sample=True,
        )
    trimmed = [
        out[len(inp):]
        for inp, out in zip(inputs['input_ids'], generated_ids)
    ]
    return vlm_processor.batch_decode(
        trimmed, skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()


def parse_thinking_output(raw_output):
    m = re.match(r'<think>\n?(.*?)</think>\n?\n?(.*)', raw_output, flags=re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip()
    return '', raw_output.strip()


def extract_verdict(answer_text):
    text  = answer_text.strip().lower()
    match = re.search(r'answer\s*:\s*(yes|no)', text)
    if match:
        return match.group(1)
    if text.startswith('yes') or 'yes' in text[:20]:
        return 'yes'
    if text.startswith('no')  or 'no'  in text[:20]:
        return 'no'
    return 'no'


def vlm_verify(image_bgr, defect_class):
    pil    = Image.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    prompt = cfg.PROMPTS[defect_class]
    raw    = vlm_query(pil, prompt, enable_thinking=False,
                       max_tokens=cfg.VLM_MAX_TOKENS_VERIFY)
    return extract_verdict(raw), raw


def vlm_get_reason(image_bgr, defect_class):
    pil    = Image.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    prompt = cfg.REASONING_PROMPTS[defect_class]
    raw    = vlm_query(pil, prompt, enable_thinking=True,
                       max_tokens=cfg.VLM_MAX_TOKENS_REASON)
    thinking, answer = parse_thinking_output(raw)
    reason = answer.strip() if answer.strip() else 'No explicit reason.'
    return thinking, reason, raw


print("VLM functions ready.")

VLM functions ready.


In [ ]:
# ──────────────────────────────────────────────────────────────
# 7.1  Inline Display Helper
# ──────────────────────────────────────────────────────────────

def show_detection(img_bgr, bbox, roi_coords, roi_bgr, label, conf,
                   shrink_info_str, verdict, reason, thinking, vlm_raw,
                   det_num, frame_name, route, track_id=None, cached=False):

    icon = {
        'AUTO-ACCEPT'    : '🟢',
        'LOW-CONF REJECT': '⚫',
        'VLM-YES'        : '🟢',
        'VLM-NO'         : '🔴',
        'CACHED-ACCEPT'  : '🔵',
        'CACHED-REJECT'  : '🟣',
    }.get(route, '🟡')

    tid_str   = f' | Track#{track_id}' if track_id is not None else ''
    cache_str = ' <i>(cached)</i>'     if cached              else ''

    display(HTML(
        f'<hr><h4>{icon} Det #{det_num} -- '
        f'<code>{frame_name}</code>{tid_str} | {label} ({conf:.3f}) '
        f'| <b>{route}</b>{cache_str}</h4>'
    ))

    x1, y1, x2, y2 = bbox
    print(f'  YOLO bbox : [{x1}, {y1}, {x2}, {y2}]')

    if route in ('AUTO-ACCEPT', 'LOW-CONF REJECT', 'CACHED-ACCEPT', 'CACHED-REJECT'):
        fig, ax = plt.subplots(1, 1, figsize=(6, 4))
        ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        color_map = {
            'AUTO-ACCEPT'    : 'lime',
            'LOW-CONF REJECT': 'gray',
            'CACHED-ACCEPT'  : 'cyan',
            'CACHED-REJECT'  : 'violet',
        }
        ax.add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            lw=2, edgecolor=color_map.get(route, 'yellow'), facecolor='none'))
        ax.set_title(route, fontsize=9)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        return

    ex1, ey1, ex2, ey2 = roi_coords
    print(f'  ROI coords: [{ex1}, {ey1}, {ex2}, {ey2}]')
    print(f'  {shrink_info_str}')

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    vis = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    axes[0].imshow(vis)
    axes[0].add_patch(patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        lw=2, edgecolor='red', facecolor='none', label='YOLO'))
    axes[0].add_patch(patches.Rectangle(
        (ex1, ey1), ex2-ex1, ey2-ey1,
        lw=2, edgecolor='orange', ls='--', facecolor='none', label='ROI'))
    axes[0].legend(fontsize=7)
    axes[0].set_title(f'YOLO: {label} ({conf:.2f})  Track#{track_id}', fontsize=9)
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB))
    axes[1].set_title('ROI -> VLM', fontsize=9)
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    v_col = '\033[92m' if verdict == 'yes' else '\033[91m'
    print(f'  VLM Verdict: {v_col}{verdict.upper()}\033[0m')
    print(f'  VLM Raw    : {vlm_raw[:200]}')
    if reason:
        print(f'  Reason     : {reason}')
    if thinking:
        snippet = thinking[:300] + ('...' if len(thinking) > 300 else '')
        print(f'  Thinking   : {snippet}')


print("Display helper ready.")

Display helper ready.


In [ ]:
# ──────────────────────────────────────────────────────────────
# 8.1  YOLO Tracking Visualization  (runs BEFORE VLM pipeline)
# ──────────────────────────────────────────────────────────────

print("Running YOLO tracking visualization pass (no VLM)...\n")

cap_vis = cv2.VideoCapture(VIDEO_PATH)
total_vis = int(cap_vis.get(cv2.CAP_PROP_FRAME_COUNT))

yolo_model.predictor = None

tracking_sample_frames = []
tracking_stats   = defaultdict(int)
unique_track_ids = set()

frame_idx_vis = 0

for _ in tqdm(range(total_vis), desc='YOLO Tracking Vis'):
    ret, frame = cap_vis.read()
    if not ret:
        break

    if frame_idx_vis % FRAME_SAMPLE_RATE != 0:
        frame_idx_vis += 1
        continue

    results = yolo_model.track(frame, persist=True, verbose=False)[0]

    if results.boxes is not None and results.boxes.id is not None:
        for r, tid in zip(results.boxes, results.boxes.id):
            cls_name = cfg.CLASS_NAMES.get(int(r.cls[0]), 'unknown')
            tracking_stats[cls_name] += 1
            unique_track_ids.add(int(tid))

    annotated = results.plot()

    if len(tracking_sample_frames) < cfg.MAX_TRACKING_FRAMES:
        tracking_sample_frames.append((annotated.copy(), frame_idx_vis))

    frame_idx_vis += 1

cap_vis.release()
yolo_model.predictor = None

print(f"\n-- YOLO Tracking Visualization Summary --")
print(f"  Unique track IDs seen : {len(unique_track_ids)}")
for cls_name, cnt in sorted(tracking_stats.items()):
    print(f"  Detections ({cls_name:<10}): {cnt}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 8.2  YOLO Tracking -- Inline Frame Grid
# ──────────────────────────────────────────────────────────────

n_show = min(len(tracking_sample_frames), 12)

if n_show == 0:
    print("No tracked frames to display -- check VIDEO_PATH.")
else:
    cols   = 3
    rows_n = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(6*cols, 4*rows_n))
    axes = np.array(axes).flatten()

    for idx in range(n_show):
        annotated_bgr, fidx = tracking_sample_frames[idx]
        axes[idx].imshow(cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(f'Frame {fidx}', fontsize=9)
        axes[idx].axis('off')

    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')

    plt.suptitle(
        f'YOLO Tracking -- {len(unique_track_ids)} unique track IDs detected\n'
        f'(boxes + IDs drawn by model.track, before VLM)',
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────
# 8.3  YOLO Tracking -- Detections per Class + Cache Benefit
# ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

classes = list(tracking_stats.keys())
counts  = [tracking_stats[c] for c in classes]
if classes:
    bars = axes[0].bar(classes, counts,
                       color=['#3498db', '#e67e22'][:len(classes)],
                       edgecolor='white', alpha=0.85)
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            axes[0].text(bar.get_x() + bar.get_width()/2, h + 0.3,
                         str(int(h)), ha='center', fontweight='bold')
axes[0].set_title('YOLO Tracking -- Total Detections per Class')
axes[0].set_ylabel('Detection Count')
axes[0].grid(axis='y', alpha=0.3)

total_det = sum(counts)
axes[1].bar(['Unique Track IDs\n(VLM calls needed)', 'Total Detections\n(without tracking)'],
            [len(unique_track_ids), total_det],
            color=['#9b59b6', '#2ecc71'], edgecolor='white', alpha=0.85)
axes[1].set_title('Unique Defects vs Total Detections\n(gap = VLM calls saved by tracking)')
axes[1].set_ylabel('Count')
axes[1].grid(axis='y', alpha=0.3)
for bar, val in zip(axes[1].patches, [len(unique_track_ids), total_det]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()
vlm_savings = total_det - len(unique_track_ids)
print(f"\n  VLM calls saved by tracking: {vlm_savings} "
      f"({vlm_savings/total_det*100:.1f}% reduction)" if total_det > 0 else "")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 9.1  Run Full YOLO-Tracking + VLM Pipeline  (sends to dashboard)
# ──────────────────────────────────────────────────────────────
# First time a track_id is seen -> route + VLM if needed -> POST to dashboard
# Subsequent frames of same track_id -> cached, no dashboard send
# ──────────────────────────────────────────────────────────────

stats = {
    'total_frames'    : 0,
    'processed_frames': 0,
    'total_detections': 0,
    'rejected_low_conf': 0,
    'auto_accepted'   : 0,
    'sent_to_vlm'     : 0,
    'vlm_accepted'    : 0,
    'vlm_rejected'    : 0,
    'cached_accepted' : 0,
    'cached_rejected' : 0,
    'dashboard_sent'  : 0,
}

track_decisions = {}
class_accepted = defaultdict(int)
class_rejected = defaultdict(int)

vis_accepted, vis_rejected = [], []
conf_auto, conf_vlm_yes, conf_vlm_no, conf_low = [], [], [], []
frame_det_counter = defaultdict(int)
det_counter = 0
frame_idx   = 0

YOLO_MODEL_NAME = os.path.basename(YOLO_WEIGHTS)

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
W_vid        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_vid        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video      : {VIDEO_PATH}")
print(f"Resolution : {W_vid} x {H_vid}")
print(f"FPS        : {fps:.2f}")
print(f"Frames     : {total_frames}")
print(f"Sampling   : every {FRAME_SAMPLE_RATE} frame(s)")
print(f"Dashboard  : {DASHBOARD_URL}\n")

for _ in tqdm(range(total_frames), desc='YOLO+VLM Pipeline'):

    ret, img = cap.read()
    if not ret:
        break

    stats['total_frames'] += 1

    if frame_idx % FRAME_SAMPLE_RATE != 0:
        frame_idx += 1
        continue

    stats['processed_frames'] += 1
    H, W       = img.shape[:2]
    frame_name = f'frame_{frame_idx:06d}.jpg'

    display(HTML(f'<h3>Frame {frame_idx:06d}  ({W}x{H})</h3>'))

    pred_boxes = get_yolo_tracked_boxes(img)
    stats['total_detections'] += len(pred_boxes)

    if len(pred_boxes) == 0:
        print('  No YOLO detections.')
        frame_idx += 1
        continue

    print(f'  YOLO: {len(pred_boxes)} det(s)  |  track IDs: {[b["track_id"] for b in pred_boxes]}')

    # Pre-compute ROIs for unseen VLM candidates
    vlm_candidates = [
        b for b in pred_boxes
        if cfg.LOW_THRESH <= b['confidence'] <= cfg.HIGH_THRESH
        and b['track_id'] not in track_decisions
    ]

    expanded_rois = []
    for box in vlm_candidates:
        x1, y1, x2, y2 = box['bbox']
        expanded_rois.append(
            list(adjust_to_square_bbox(x1, y1, x2, y2, W, H, cfg.INITIAL_EXPAND_SCALE))
        )

    final_rois, shrink_infos = [], []
    for i, (box, roi) in enumerate(zip(vlm_candidates, expanded_rois)):
        overlap_found = any(
            overlap_ratio(roi, expanded_rois[j]) > cfg.OVERLAP_THRESH
            for j in range(len(expanded_rois)) if j != i
        )
        if overlap_found:
            x1, y1, x2, y2 = box['bbox']
            ex1, ey1, ex2, ey2 = adaptive_expand_bbox(x1, y1, x2, y2, W, H, expanded_rois, i)
            final_rois.append([ex1, ey1, ex2, ey2])
            bbw, bbh = x2-x1, y2-y1
            actual   = max((ex2-ex1)/(bbw+1e-6), (ey2-ey1)/(bbh+1e-6))
            shrink_infos.append(f'Adaptive Shrink: {cfg.INITIAL_EXPAND_SCALE}x -> {actual:.2f}x')
        else:
            final_rois.append(roi)
            shrink_infos.append(f'Expanded: {cfg.INITIAL_EXPAND_SCALE}x (no overlap)')

    vlm_idx = 0

    for box in pred_boxes:
        det_counter += 1
        x1, y1, x2, y2 = box['bbox']
        conf     = box['confidence']
        label    = box['class']
        cls_id   = box['class_id']
        track_id = box['track_id']

        # ==================================================
        # CACHED VERDICT -- track_id seen before (no dashboard send)
        # ==================================================
        if track_id in track_decisions:
            cached = track_decisions[track_id]
            verdict = cached['verdict']

            if verdict in ('yes', 'auto'):
                stats['cached_accepted'] += 1
                show_detection(img, [x1,y1,x2,y2], None, None, label, conf,
                               '', verdict, None, '', '',
                               det_counter, frame_name, 'CACHED-ACCEPT',
                               track_id=track_id, cached=True)
            else:
                stats['cached_rejected'] += 1
                show_detection(img, [x1,y1,x2,y2], None, None, label, conf,
                               '', verdict, cached.get('reason'), '', '',
                               det_counter, frame_name, 'CACHED-REJECT',
                               track_id=track_id, cached=True)
            continue

        # ==================================================
        # FIRST TIME seeing this track_id -> decide + send to dashboard
        # ==================================================

        # Crop around detection for the dashboard image
        det_crop = crop_bbox(img, [x1, y1, x2, y2])

        # ROUTE A: conf < 0.3 -> REJECT (low confidence)
        if conf < cfg.LOW_THRESH:
            stats['rejected_low_conf'] += 1
            conf_low.append(conf)
            class_rejected[label] += 1
            track_decisions[track_id] = {
                'verdict': 'low', 'label': label, 'class_id': cls_id,
                'conf_first': conf, 'reason': 'Low confidence rejection',
                'thinking': '', 'vlm_raw': '',
            }
            # Send to dashboard as a YOLO detection
            send_yolo_detection(label, conf, YOLO_MODEL_NAME, det_crop)
            stats['dashboard_sent'] += 1
            show_detection(img, [x1,y1,x2,y2], None, None, label, conf,
                           '', 'rejected', None, '', '',
                           det_counter, frame_name, 'LOW-CONF REJECT',
                           track_id=track_id)
            continue

        # ROUTE B: conf > 0.8 -> AUTO-ACCEPT
        if conf > cfg.HIGH_THRESH:
            stats['auto_accepted'] += 1
            conf_auto.append(conf)
            class_accepted[label] += 1
            track_decisions[track_id] = {
                'verdict': 'auto', 'label': label, 'class_id': cls_id,
                'conf_first': conf, 'reason': '', 'thinking': '', 'vlm_raw': '',
            }
            send_yolo_detection(label, conf, YOLO_MODEL_NAME, det_crop)
            stats['dashboard_sent'] += 1
            if len(vis_accepted) < cfg.MAX_VIS_PER_CATEGORY:
                vis_accepted.append((img.copy(), [x1,y1,x2,y2], label, conf, 'auto'))
            show_detection(img, [x1,y1,x2,y2], None, None, label, conf,
                           '', 'accepted', None, '', '',
                           det_counter, frame_name, 'AUTO-ACCEPT',
                           track_id=track_id)
            continue

        # ROUTE C: 0.3 <= conf <= 0.8 -> VLM verify
        stats['sent_to_vlm'] += 1
        roi_coords = final_rois[vlm_idx]
        shrink_str = shrink_infos[vlm_idx]
        vlm_idx   += 1

        ex1, ey1, ex2, ey2 = roi_coords
        roi = img[ey1:ey2, ex1:ex2]
        if roi.size == 0:
            continue

        verdict, vlm_raw = vlm_verify(roi, label)

        if verdict == 'yes':
            # VLM confirmed -> send YOLO detection to dashboard
            stats['vlm_accepted'] += 1
            conf_vlm_yes.append(conf)
            class_accepted[label] += 1
            track_decisions[track_id] = {
                'verdict': 'yes', 'label': label, 'class_id': cls_id,
                'conf_first': conf, 'reason': '', 'thinking': '', 'vlm_raw': vlm_raw,
            }
            send_yolo_detection(label, conf, YOLO_MODEL_NAME, det_crop)
            stats['dashboard_sent'] += 1
            if len(vis_accepted) < cfg.MAX_VIS_PER_CATEGORY:
                vis_accepted.append((img.copy(), [x1,y1,x2,y2], label, conf, 'vlm'))
            show_detection(img, [x1,y1,x2,y2], roi_coords, roi, label, conf,
                           shrink_str, verdict, None, '', vlm_raw,
                           det_counter, frame_name, 'VLM-YES',
                           track_id=track_id)

        else:
            # VLM rejected -> send YOLO detection + VLM-NO to dashboard
            stats['vlm_rejected'] += 1
            conf_vlm_no.append(conf)
            class_rejected[label] += 1
            thinking, reason, reason_raw = vlm_get_reason(roi, label)
            track_decisions[track_id] = {
                'verdict': 'no', 'label': label, 'class_id': cls_id,
                'conf_first': conf, 'reason': reason,
                'thinking': thinking, 'vlm_raw': vlm_raw,
            }
            # Send YOLO detection first, then VLM rejection with the detection_id
            det_id = send_yolo_detection(label, conf, YOLO_MODEL_NAME, det_crop)
            send_vlm_no(label, VLM_MODEL_ID, reason, roi, detection_id=det_id)
            stats['dashboard_sent'] += 2
            if len(vis_rejected) < cfg.MAX_VIS_PER_CATEGORY:
                vis_rejected.append((img.copy(), roi.copy(),
                                     [x1,y1,x2,y2], label, conf, reason))
            show_detection(img, [x1,y1,x2,y2], roi_coords, roi, label, conf,
                           shrink_str, verdict, reason, thinking, vlm_raw,
                           det_counter, frame_name, 'VLM-NO',
                           track_id=track_id)

    frame_idx += 1

cap.release()
clear_gpu()
print('\nPipeline complete.')
print(f'  Dashboard requests sent: {stats["dashboard_sent"]}')

In [ ]:
# ──────────────────────────────────────────────────────────────
# 10.1  Pipeline Summary
# ──────────────────────────────────────────────────────────────

total_accepted = stats['auto_accepted'] + stats['vlm_accepted']
total_rejected = stats['rejected_low_conf'] + stats['vlm_rejected']
total_cached   = stats['cached_accepted'] + stats['cached_rejected']
vlm_acc_rate   = (stats['vlm_accepted'] / stats['sent_to_vlm'] * 100
                  if stats['sent_to_vlm'] > 0 else 0)

print('=' * 65)
print('  YOLO TRACKING + VLM PIPELINE -- DASHBOARD SUMMARY')
print('=' * 65)
print(f'\n  Video                      : {VIDEO_PATH}')
print(f'  Dashboard                  : {DASHBOARD_URL}')
print(f'  Total frames               : {stats["total_frames"]}')
print(f'  Processed (1/{FRAME_SAMPLE_RATE})            : {stats["processed_frames"]}')
print(f'  Total YOLO detections      : {stats["total_detections"]}')
print(f'  Unique track IDs           : {len(track_decisions)}')
print(f'\n  -- Routing (first-time per track_id) --')
print(f'  Rejected   (conf < {cfg.LOW_THRESH})      : {stats["rejected_low_conf"]}')
print(f'  Auto-accept (conf > {cfg.HIGH_THRESH})     : {stats["auto_accepted"]}')
print(f'  Sent to VLM (mid zone)     : {stats["sent_to_vlm"]}')
print(f'\n  -- VLM Decisions --')
print(f'  VLM YES (accepted)         : {stats["vlm_accepted"]}')
print(f'  VLM NO  (rejected)         : {stats["vlm_rejected"]}')
print(f'  VLM acceptance rate        : {vlm_acc_rate:.1f}%')
print(f'\n  -- Tracking Cache Hits --')
print(f'  Cached-accept applied      : {stats["cached_accepted"]}')
print(f'  Cached-reject applied      : {stats["cached_rejected"]}')
print(f'  VLM calls saved by cache   : {total_cached}')
print(f'\n  -- Dashboard --')
print(f'  HTTP requests sent         : {stats["dashboard_sent"]}')
print(f'  Total accepted (unique)    : {total_accepted}')
print(f'  Total rejected (unique)    : {total_rejected}')
print('=' * 65)

In [ ]:
# ──────────────────────────────────────────────────────────────
# 11.1  Visualization -- Detection Routing Breakdown
# ──────────────────────────────────────────────────────────────

labels_pie = ['Low-Conf\nReject', 'Auto-Accept\n(>0.8)', 'VLM Accept',
              'VLM Reject', 'Cached Accept', 'Cached Reject']
sizes      = [stats['rejected_low_conf'], stats['auto_accepted'],
              stats['vlm_accepted'],      stats['vlm_rejected'],
              stats['cached_accepted'],   stats['cached_rejected']]
colors_pie = ['#95a5a6', '#2ecc71', '#27ae60', '#e74c3c', '#3498db', '#9b59b6']
explode    = (0.03, 0.03, 0.03, 0.08, 0.03, 0.03)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

nonzero = [(l, s, c, e) for l, s, c, e in zip(labels_pie, sizes, colors_pie, explode) if s > 0]
if nonzero:
    ax1.pie([x[1] for x in nonzero], explode=[x[3] for x in nonzero],
            labels=[x[0] for x in nonzero], colors=[x[2] for x in nonzero],
            autopct=lambda p: f'{p:.1f}%\n({int(round(p*sum(sizes)/100))})',
            startangle=90, textprops={'fontsize': 8})
ax1.set_title('Detection Routing Breakdown')

ax2.bar(labels_pie, sizes, color=colors_pie, edgecolor='white')
for i, v in enumerate(sizes):
    ax2.text(i, v + 0.3, str(v), ha='center', fontweight='bold', fontsize=9)
ax2.set_ylabel('Count')
ax2.set_title('Detection Counts by Route')
ax2.tick_params(axis='x', labelsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────
# 11.2  Visualization -- Confidence Distributions
# ──────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
configs = [
    (conf_auto,    '#2ecc71', f'Auto-Accept >0.8 (n={len(conf_auto)})',    cfg.HIGH_THRESH, 'blue', f'HIGH={cfg.HIGH_THRESH}'),
    (conf_vlm_yes, '#27ae60', f'VLM-YES (n={len(conf_vlm_yes)})',           None,            None,   None),
    (conf_vlm_no,  '#e74c3c', f'VLM-NO  (n={len(conf_vlm_no)})',            None,            None,   None),
    (conf_low,     '#95a5a6', f'Low-Conf Reject <0.3 (n={len(conf_low)})',  cfg.LOW_THRESH,  'red',  f'LOW={cfg.LOW_THRESH}'),
]
for ax, (data, color, title, thresh, tcolor, tlabel) in zip(axes, configs):
    if data:
        ax.hist(data, bins=20, color=color, edgecolor='white', alpha=0.85)
    if thresh is not None:
        ax.axvline(thresh, color=tcolor, ls='--', label=tlabel)
        ax.legend(fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Count')
    ax.set_xlim(0, 1)

plt.suptitle('Confidence Distribution by Decision Category', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ──────────────────────────────────────────────────────────────
# 11.3  Visualization -- Sample Accepted Detections
# ──────────────────────────────────────────────────────────────

if vis_accepted:
    n      = min(len(vis_accepted), 8)
    cols   = min(n, 4)
    rows_n = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(5*cols, 4.5*rows_n))
    if rows_n == 1 and cols == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()
    for idx in range(n):
        img_vis, bbox, cls, conf_v, method = vis_accepted[idx]
        x1, y1, x2, y2 = bbox
        axes[idx].imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
        axes[idx].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, lw=2, edgecolor='lime', facecolor='none'))
        axes[idx].set_title(f'  {cls} ({conf_v:.2f}) [{method}]', fontsize=9)
        axes[idx].axis('off')
    for idx in range(n, len(axes)):
        axes[idx].axis('off')
    plt.suptitle('Sample Accepted Detections', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No accepted detections to display.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 11.4  Visualization -- VLM Rejected Detections + Reasoning
# ──────────────────────────────────────────────────────────────

if vis_rejected:
    n   = min(len(vis_rejected), 8)
    fig, axes = plt.subplots(n, 2, figsize=(12, 4*n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for idx in range(n):
        img_vis, roi_vis, bbox, cls, conf_v, reason = vis_rejected[idx]
        x1, y1, x2, y2 = bbox
        axes[idx, 0].imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
        axes[idx, 0].add_patch(patches.Rectangle(
            (x1, y1), x2-x1, y2-y1, lw=2, edgecolor='red', facecolor='none'))
        axes[idx, 0].set_title(f'  {cls} ({conf_v:.2f}) -> VLM: NO', fontsize=9)
        axes[idx, 0].axis('off')
        axes[idx, 1].imshow(cv2.cvtColor(roi_vis, cv2.COLOR_BGR2RGB))
        reason_short = reason[:80] + '...' if reason and len(reason) > 80 else (reason or '')
        axes[idx, 1].set_title(f'Reason: {reason_short}', fontsize=8, wrap=True)
        axes[idx, 1].axis('off')
    plt.suptitle('VLM Rejected Detections + Reasoning', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No VLM-rejected samples to display.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 11.5  Visualization -- Accepted vs Rejected by Class
# ──────────────────────────────────────────────────────────────

all_classes = sorted(set(list(class_accepted.keys()) + list(class_rejected.keys())))

if all_classes:
    x = np.arange(len(all_classes))
    w = 0.35
    fig, ax = plt.subplots(figsize=(8, 5))
    bars1 = ax.bar(x - w/2, [class_accepted.get(c, 0) for c in all_classes],
                   w, label='Accepted', color='#2ecc71', alpha=0.85)
    bars2 = ax.bar(x + w/2, [class_rejected.get(c, 0) for c in all_classes],
                   w, label='Rejected', color='#e74c3c', alpha=0.85)
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
                    str(int(h)), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(all_classes, fontsize=11)
    ax.set_ylabel('Detection Count')
    ax.set_title('Accepted vs Rejected Detections by Class')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No detection data to display.")

In [ ]:
# ──────────────────────────────────────────────────────────────
# 12.1  All Done
# ───────────���──────────────────────────────────────────────────

print("\n  Pipeline complete -- all results sent to dashboard.")
print(f"\n  Dashboard URL : {DASHBOARD_URL}")
print(f"  Open in browser to view detections and VLM rejections.")
print(f"\n  HTTP requests sent : {stats['dashboard_sent']}")
print(f"  Unique track IDs   : {len(track_decisions)}")